## Statistical Inference

In this section, we move beyond and apply statistical inference to evaluate whether the patterns\
observed in stock returns are likely to reflect real underlying effects rather than random variation.

### Target Population

The target population in this analysis is the distribution of daily stock returns for the selected\
stocks in this project (AAPL, MSFT, AMZN, GOOGL, META, JPM, JNJ, XOM, PG, HD) over the studied period. Since we have only historical daily\
returns from 2010 onward, our dataset is treated as a sample from the broader process that generates\
stock returns over time.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats

In [2]:
data_path = Path("../02_magic/01_data/raw/stock_prices.csv")
df = pd.read_csv(data_path)

print("Shape:", df.shape)
df.head()

Shape: (40351, 8)


,Date,Open,High,Low,Close,Adj Close,Volume,Ticker
0,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.412385,493729600,AAPL
1,2010-01-05,7.664286,7.699643,7.616071,7.656429,6.423469,601904800,AAPL
2,2010-01-06,7.656429,7.686786,7.526786,7.534643,6.321294,552160000,AAPL
3,2010-01-07,7.562500,7.571429,7.466071,7.520714,6.309608,477131200,AAPL
4,2010-01-08,7.510714,7.571429,7.466429,7.570714,6.351558,447610800,AAPL


In [3]:
df_inf = df.copy()

df_inf["Date"] = pd.to_datetime(df_inf["Date"])
df_inf["Return_1d"] = df_inf.groupby("Ticker")["Close"].pct_change()

df_inf = df_inf.dropna(subset=["Return_1d"]).copy()

alpha = 0.05

df_inf[["Date", "Ticker", "Close", "Return_1d"]].head()

,Date,Ticker,Close,Return_1d
1,2010-01-05,AAPL,7.656429,0.001729
2,2010-01-06,AAPL,7.534643,-0.015906
3,2010-01-07,AAPL,7.520714,-0.001849
4,2010-01-08,AAPL,7.570714,0.006648
5,2010-01-11,AAPL,7.503929,-0.008821


### Confidence Intervals for Mean Daily Returns

As a first step, I calculate 95% confidence intervals for the mean daily return of each stock.\
This shows the range in which the true average return is likely to fall based on the observed data.

In [4]:
ci_results = []

for ticker in df_inf["Ticker"].unique():
    sample = df_inf.loc[df_inf["Ticker"] == ticker, "Return_1d"].dropna()
    
    mean_ret = sample.mean()
    sem = stats.sem(sample)
    n = len(sample)

    ci_low, ci_high = stats.t.interval(
        confidence=0.95,
        df=n - 1,
        loc=mean_ret,
        scale=sem
    )

    ci_results.append({
        "Ticker": ticker,
        "Mean_Return": mean_ret,
        "CI_95_Lower": ci_low,
        "CI_95_Upper": ci_high
    })

ci_df = pd.DataFrame(ci_results).sort_values("Mean_Return", ascending=False)
ci_df.round(6)

,Ticker,Mean_Return,CI_95_Lower,CI_95_Upper
6,META,0.001134,0.000303,0.001964
1,AMZN,0.001096,0.000463,0.001730
0,AAPL,0.001025,0.000482,0.001567
2,GOOGL,0.000901,0.000367,0.001435
7,MSFT,0.000763,0.000267,0.001258
3,HD,0.000712,0.000262,0.001162
5,JPM,0.000631,0.000099,0.001163
4,JNJ,0.000376,0.000049,0.000703
9,XOM,0.000313,-0.000172,0.000797
8,PG,0.000268,-0.000067,0.000603


### Confidence Interval Results

In the current results, most stocks have 95% confidence intervals above zero, which suggests\
that their average daily returns were positive over the observed period. However, some intervals still\
touch zero, so the evidence is not equally strong for every stock.

### Hypothesis 1: Mean Daily Return vs 0

Next, I test whether the average daily return of each stock is different from 0 using a one-sample t-test.

- **Null hypothesis ($H_0$):** the mean daily return is equal to 0  
- **Alternative hypothesis ($H_1$):** the mean daily return is different from 0  

The significance level is set to 0.05.

In [5]:
ttest_zero_results = []

for ticker in df_inf["Ticker"].unique():
    sample = df_inf.loc[df_inf["Ticker"] == ticker, "Return_1d"].dropna()
    t_stat, p_value = stats.ttest_1samp(sample, popmean=0)

    ttest_zero_results.append({
        "Ticker": ticker,
        "Mean_Return": sample.mean(),
        "T_Statistic": t_stat,
        "P_Value": p_value,
        "Reject_H0_at_5pct": p_value < alpha
    })

ttest_zero_df = pd.DataFrame(ttest_zero_results).sort_values("P_Value")
ttest_zero_df.round(6)

,Ticker,Mean_Return,T_Statistic,P_Value,Reject_H0_at_5pct
0,AAPL,0.001025,3.700628,0.000218,True
1,AMZN,0.001096,3.391693,0.000701,True
2,GOOGL,0.000901,3.307656,0.000949,True
3,HD,0.000712,3.102985,0.001929,True
7,MSFT,0.000763,3.018057,0.002560,True
6,META,0.001134,2.675510,0.007496,True
5,JPM,0.000631,2.323816,0.020184,True
4,JNJ,0.000376,2.254392,0.024224,True
8,PG,0.000268,1.568121,0.116930,False
9,XOM,0.000313,1.265173,0.205881,False


### Hypothesis 1 Results

In the current results, most stocks have p-values below 0.05, so the **null hypothesis is rejected** for most tickers.\
However, **PG** and **XOM** are above the 5% threshold, so for those two stocks the test does not show a statistically significant difference from 0.\
Average returns are still quite small, so the result is statistically meaningful but modest in size.

### Hypothesis 2: AAPL vs MSFT Mean Returns

Here, I compare whether **AAPL** and **MSFT** have the same average daily return using a two-sample t-test.

- **Null hypothesis ($H_0$):** AAPL and MSFT have the same mean daily return  
- **Alternative hypothesis ($H_1$):** AAPL and MSFT have different mean daily returns  

The significance level is set to 0.05.

In [6]:
stock_1 = "AAPL"
stock_2 = "MSFT"

sample_1 = df_inf.loc[df_inf["Ticker"] == stock_1, "Return_1d"].dropna()
sample_2 = df_inf.loc[df_inf["Ticker"] == stock_2, "Return_1d"].dropna()

t_stat, p_value = stats.ttest_ind(sample_1, sample_2, equal_var=False)

stock_comparison_df = pd.DataFrame({
    "Stock_1": [stock_1],
    "Stock_2": [stock_2],
    "Mean_1": [sample_1.mean()],
    "Mean_2": [sample_2.mean()],
    "T_Statistic": [t_stat],
    "P_Value": [p_value],
    "Reject_H0_at_5pct": [p_value < alpha]
})

stock_comparison_df.round(6)

,Stock_1,Stock_2,Mean_1,Mean_2,T_Statistic,P_Value,Reject_H0_at_5pct
0,AAPL,MSFT,0.001025,0.000763,0.69848,0.484897,False


### Hypothesis 2 Results

The p-value is **above** 0.05, so the null hypothesis is not rejected.\
The difference between the average daily returns of **AAPL** and **MSFT** **is not statistically significant** in this sample.\
Although **AAPL** has a slightly higher mean return, the gap is too small to conclude that the two stocks behave differently on average.

### Hypothesis 3: Crisis vs Non-Crisis Returns

Here, I compare average daily returns during the 2020 crash period and the rest of the sample using a two-sample t-test.

- **Crisis period:** 2020-02-20 to 2020-04-30  
- **Non-crisis period:** all other dates  
- **Null hypothesis ($H_0$):** the mean daily return is the same in both periods  
- **Alternative hypothesis ($H_1$):** the mean daily return is different  

The significance level is set to 0.05.

In [7]:
crisis_start = "2020-02-20"
crisis_end = "2020-04-30"

df_inf["Period"] = np.where(
    (df_inf["Date"] >= crisis_start) & (df_inf["Date"] <= crisis_end),
    "Crisis",
    "Non-Crisis"
)

crisis_returns = df_inf.loc[df_inf["Period"] == "Crisis", "Return_1d"].dropna()
non_crisis_returns = df_inf.loc[df_inf["Period"] == "Non-Crisis", "Return_1d"].dropna()

t_stat, p_value = stats.ttest_ind(crisis_returns, non_crisis_returns, equal_var=False)

period_comparison_df = pd.DataFrame({
    "Crisis_Mean_Return": [crisis_returns.mean()],
    "Non_Crisis_Mean_Return": [non_crisis_returns.mean()],
    "T_Statistic": [t_stat],
    "P_Value": [p_value],
    "Reject_H0_at_5pct": [p_value < alpha]
})

period_comparison_df.round(6)

,Crisis_Mean_Return,Non_Crisis_Mean_Return,T_Statistic,P_Value,Reject_H0_at_5pct
0,-0.000809,0.000735,-0.724069,0.469362,False


### Hypothesis 3 Results

The p-value is well above 0.05, so the **null hypothesis is not rejected**.\
This means the average daily return during the 2020 crash period was not statistically different\
from the rest of the sample in this test. Although the crisis-period mean return is lower, the difference\
is too small relative to the variability in returns to be considered significant.